# Laboratori 02: Aritmetika kompjuterike dhe analiza e gabimeve

**Lënda:** Simulim dhe Metoda Numerike  
**Lidhja me leksionin:** Kapitulli 2  
**Kohëzgjatja e rekomanduar:** 2 orë  
**Botimi:** Versioni i studentit

> **Pyetja udhëheqëse:** Si dallohen gabimi i algoritmit, gabimi i rrumbullakimit dhe keqkushtëzimi i vetë problemit?

Notebook-u ndahet në dy gjysma të barasvlershme. Gjysma e parë përdor Python; gjysma e dytë përdor MATLAB/Octave. Studenti mund të zgjedhë njërën gjuhë, por duhet të raportojë të njëjtat kontrolle numerike dhe interpretime fizike.


## Rezultatet e të nxënit

Në përfundim, studenti duhet të jetë në gjendje:

- të inspektojë paraqitjen me presje lëvizëse dhe epsilonin e makinës;
- të dallojë gabimin absolut, relativ, të përparmë dhe të prapëm;
- të demonstrojë anulimin katastrofik dhe përmbledhjen e qëndrueshme;
- të lidhë numrin e kushtëzimit me ndjeshmërinë e një sistemi linear;


## Përgatitja teorike


Një numër real ruhet në trajtën e idealizuar $x=\pm m\beta^e$, me bazë $\beta$, mantisë $m$ dhe eksponent $e$. Në aritmetikën IEEE 754, veprimi elementar modelohet nga

$$
\operatorname{fl}(x\circ y)=(x\circ y)(1+\delta), \qquad |\delta|\le u,
$$

ku $u$ është njësia e rrumbullakimit. Ky model nuk thotë se çdo përgjigje ka gabim të vogël: anulimi, numri i madh i veprimeve dhe kushtëzimi mund ta përforcojnë perturbimin.

Për $f(x)$, numri relativ i kushtëzimit lokal është

$$
\kappa_f(x)=\left|\frac{x f'(x)}{f(x)}\right|.
$$

Për sistemin $A\mathbf{x}=\mathbf{b}$ përdoret $\kappa_2(A)=\|A\|_2\|A^{-1}\|_2$. Një algoritëm i qëndrueshëm prapa prodhon zgjidhjen e saktë të një problemi pak të perturbur; ai nuk mund të korrigjojë një problem thelbësisht të keqkushtëzuar.


## Algoritmi dhe kriteret e verifikimit


1. Gjejmë kufirin ku $1+\varepsilon$ nuk dallohet më nga $1$.
2. Krahasojmë formulën e drejtpërdrejtë të $1-\cos x$ me formulën e qëndrueshme $2\sin^2(x/2)$.
3. Krahasojmë shumën standarde, shumën e Kahan-it dhe një vlerë reference.
4. Ndërtojmë $A_\varepsilon=\begin{pmatrix}1&1\\1&1+\varepsilon\end{pmatrix}$ dhe masim $\kappa_2(A_\varepsilon)$, mbetjen dhe gabimin e përparmë.


Në çdo eksperiment raportohen parametrat, madhësia e rrjetës ose e mostrës, kriteri i ndalimit, një mbetje ose ligj ruajtjeje dhe një vlerësim i gabimit.


# Pjesa I — Python

Ekzekutoni qelizat sipas radhës. Ndryshoni vetëm një faktor në çdo eksperiment numerik dhe ruani figurat që mbështesin përfundimin.


### Eksperimenti Python 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(2026)


### Eksperimenti Python 2


In [ ]:
eps = 1.0
while 1.0 + eps / 2.0 > 1.0:
    eps /= 2.0
print("Epsiloni i gjetur:", eps)
print("Epsiloni i NumPy:", np.finfo(float).eps)


### Eksperimenti Python 3


In [ ]:
x = np.logspace(-16, -1, 240)
drejt = 1.0 - np.cos(x)
qendrueshem = 2.0 * np.sin(x / 2.0) ** 2
reference = 0.5 * x**2

fig, ax = plt.subplots()
ax.loglog(x, np.abs(drejt - reference), label="formula e drejtpërdrejtë")
ax.loglog(x, np.abs(qendrueshem - reference), label="formula e qëndrueshme")
ax.set(xlabel="$x$", ylabel="gabimi absolut", title="Anulimi në $1-\\cos x$")
ax.legend(); plt.show()


### Eksperimenti Python 4


In [ ]:
def shuma_kahan(v):
    # TODO: plotësoni zbatimin duke u mbështetur te algoritmi i mësipërm.
    raise NotImplementedError('Plotësoni këtë pjesë')

v = np.r_[1e16, np.ones(100_000), -1e16]
print("Shuma standarde:", np.sum(v))
print("Shuma Kahan:", shuma_kahan(v))
print("Vlera e pritshme:", 100_000.0)


### Eksperimenti Python 5


In [ ]:
eps_values = np.logspace(-1, -14, 50)
kappa, err_fwd, residual = [], [], []
x_exact = np.array([1.0, 1.0])
for e in eps_values:
    A = np.array([[1.0, 1.0], [1.0, 1.0 + e]])
    b = A @ x_exact
    x_num = np.linalg.solve(A, b)
    kappa.append(np.linalg.cond(A, 2))
    err_fwd.append(np.linalg.norm(x_num - x_exact) / np.linalg.norm(x_exact))
    residual.append(np.linalg.norm(b - A @ x_num) / np.linalg.norm(b))

fig, ax = plt.subplots()
ax.loglog(eps_values, kappa, label="$\\kappa_2(A)$")
ax.loglog(eps_values, err_fwd, label="gabimi i përparmë")
ax.loglog(eps_values, residual, label="mbetja relative")
ax.invert_xaxis(); ax.set(xlabel="$\\varepsilon$", ylabel="madhësia", title="Kushtëzimi dhe gabimet")
ax.legend(); plt.show()


## Interpretimi i rezultateve të Python-it

Përgjigjuni me fjali të plota:

1. Cili rezultat verifikon drejtpërdrejt algoritmin?
2. Në cilin regjim rezultati është i pavarur nga hapi, madhësia e mostrës ose gjendja fillestare?
3. Cili është burimi dominues i gabimit ose i pacaktueshmërisë?
4. A mbështetet përfundimi fizik nga figura dhe nga një madhësi numerike?


# Pjesa II — MATLAB/Octave

Qelizat vijuese janë të tipit `raw` që notebook-u të mbetet i hapshëm me kernelin Python. Kopjojini në MATLAB ose Octave, ose ndryshojini në qeliza kodi kur përdorni një kernel MATLAB/Octave.


### Eksperimenti MATLAB/Octave 1


## Krahasimi ndërmjet gjuhëve

Kontrolloni që Python dhe MATLAB/Octave japin rezultate në përputhje me tolerancën e arsyeshme. Ndryshimet në paraqitjen grafike nuk janë dallime shkencore; dallim shkencor është ndryshimi i vlerës, rendit të konvergjencës, mbetjes, energjisë ose intervalit të besimit.


## Produkti që dorëzohet

Dorëzimi duhet të përmbajë:

- notebook-un e ekzekutuar ose skriptin MATLAB/Octave;
- figurat me boshte, njësi dhe legjendë;
- një tabelë të shkurtër me parametrat dhe rezultatet;
- një kontroll verifikimi;
- një paragraf interpretimi fizik;
- një paragraf mbi kufizimet dhe burimet e gabimit;
- funksionet e shkruara nga studenti, të organizuara në mënyrë që të mund të përfshihen më vonë në paketën e tij/saj.


## Kontroll i shpejtë

- A përdor kodi farë të dokumentuar kur ka rastësi?
- A dallohen gabimi numerik dhe pacaktueshmëria e hyrjeve?
- A janë ruajtur njësitë fizike?
- A është kontrolluar rezultati me një kufi, zgjidhje reference, mbetje ose ligj ruajtjeje?
- A mund të riprodhohet rezultati nga një student tjetër?
